# MatchSense walkthrough: Strands Decider 2B decides, Amazon Nova speaks only when tactics change

`#strands-decider-2b` `#policy` · Repo: https://github.com/Poobalan1210/agentic-football

This notebook walks through the decision handed to **Strands Decider 2B** and the reusable `#policy` pattern, using the **real recorded runs committed in the repo**. It needs **no API keys**: every Decider answer and Bedrock number below was recorded in a real run on 2026-10-24 (Decider v19 local on an Apple M4, Amazon Nova on Bedrock us-east-1). An optional last cell makes live calls if you have a Decider key and Bedrock access.

**Run it** (locally, Jupyter, or Amazon SageMaker): clone the repo, then `pip install -r requirements.txt && pip install -e .` and run all cells.

In [1]:
# Uncomment on a fresh SageMaker / Colab-style kernel:
# %pip install -q -r ../requirements.txt && %pip install -q -e ..
import json, os, statistics, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "matchsense").exists())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("APP_MODE", "mock")   # nothing below calls a model unless you run the live cell
from IPython.display import Markdown, display
print("repo root:", ROOT.name)

repo root: agentic-football


## 1. The decision handed to Decider: one typed `choice` question

Five options, each written as *"when [condition], so we [action]"*. This wording raised Decider's agreement with our answer key from 18/57 to 30/57 on a probe set (see `docs/BENCHMARK_METHODOLOGY.md`).

In [2]:
from matchsense.policy_decision import QUESTION, POLICY_OPTIONS
print(QUESTION, "\n")
for name, definition in POLICY_OPTIONS.items():
    print(f"{name:16} {definition}")

Given the current five-a-side match situation, which tactical policy should the team follow next? 

HIGH_PRESS       the opponent has the ball deep in their own half, far from our goal, and we press them high up the pitch to force a turnover
COUNTERATTACK    we have just won the ball and their defence is out of position, so we attack the open space at speed
KEEP_POSSESSION  we have the ball but their defenders are in place, so we pass safely and build patiently
PROTECT_LEAD     we are ahead near the end of the match, so we take no risks and run down the clock
LOW_BLOCK        the opponent has the ball in our half or is breaking towards our goal, so we drop deep, protect the centre and break out with controlled transitions


## 2. The state becomes sentences

A 2B model is weak at geometry from raw coordinates, so code computes the facts (who is goal-side, open space, pressure) and writes them out. Here is moment 39 of the *late comeback* match: the structured state, then the text Decider actually reads.

In [3]:
from matchsense.schemas import MatchState
from matchsense.state_text import state_to_text

run = json.loads((ROOT / "docs/results/replays/bench_cli_20261024_023923_m1_late_comeback_1.json").read_text())
moment = run["ticks"][38]
state = MatchState.model_validate(moment["state"])
print("score", state.score_us, "-", state.score_them, "| time left", state.time_remaining_s, "s | ball", state.ball, "| possession", state.possession)
print()
print(state_to_text(state))

score 0 - 1 | time left 44 s | ball x=26.0 y=10.0 | possession us

We are losing 0-1 with 44 seconds left in a 2-minute match. We have the ball in our own half, 14 m from our goal. We won the ball back just now. One opposing outfield player is between the ball and their goal. Their defence is out of position and there is open space in front of us. All our team-mates are marked. We have no open passing options. Three opponents are within 6 m of the ball. Their outfield players are pushed up, leaving space behind them. Our players are moderately tired. Objective: exploit the open space quickly.


## 3. What Decider returned at that moment (recorded)

One forward pass: a choice, a probability for every option and Decider's own confidence. No text is generated.

In [4]:
d = moment["lanes"]["A"]["decision"]
print("choice     :", d["raw_decider_policy"])
print("confidence :", d["decider_confidence"])
for k, v in sorted(d["decider_probabilities"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:16} {v:.4f}  {'#' * round(v * 40)}")
print("round trip :", d["decider_latency_ms"], "ms |", d["decider_input_tokens"], "input tokens")
print("answer key :", moment["reference_policy"])

choice     : COUNTERATTACK
confidence : 0.8557
  COUNTERATTACK    0.8846  ###################################
  HIGH_PRESS       0.0979  ####
  KEEP_POSSESSION  0.0076  
  LOW_BLOCK        0.0051  
  PROTECT_LEAD     0.0048  
round trip : 361.3 ms | 333 input tokens
answer key : COUNTERATTACK


## 4. The reusable wiring: `DeciderPolicyGate`, a Strands `InterventionHandler`

The gate runs in `before_invocation`, before any model call. If Decider's policy is unchanged and a plan already exists for it, it returns `Deny` and the agent never reaches Bedrock. If the policy changed, it returns `Transform` and tells the LLM which policy to plan for.

Below, the gate is driven by the **recorded** Decider answers for all 60 moments of the match (a stand-in for `DeciderClient` that replays them), so you can see exactly how often the LLM would be called.

In [5]:
from types import SimpleNamespace
from strands.interventions import Deny, Transform
from matchsense.decider_gate import DeciderPolicyGate
from matchsense.schemas import DeciderResult

class RecordedDecider:
    """Replays the real Decider answers from the committed run, one per call."""
    def __init__(self, ticks):
        self.answers = [t["lanes"]["A"]["decision"] for t in ticks]
        self.i = 0
    def decide_choice(self, state_text, question, options):
        a = self.answers[self.i]; self.i += 1
        return DeciderResult(choice=a["raw_decider_policy"], confidence=a["decider_confidence"],
                             probabilities=a["decider_probabilities"], client_latency_ms=a["decider_latency_ms"])

gate = DeciderPolicyGate(RecordedDecider(run["ticks"]), QUESTION, POLICY_OPTIONS)
timeline = []
for t in run["ticks"]:
    ev = SimpleNamespace(invocation_state={"state_text": t["state_text"]},
                         messages=[{"role": "user", "content": [{"text": "Write the plan."}]}])
    action = gate.before_invocation(ev)
    if isinstance(action, Transform):          # the LLM would be called: count it as a successful plan
        gate.after_model_call(SimpleNamespace(exception=None))
    timeline.append("P" if isinstance(action, Transform) else ".")

print("".join(timeline), "  (P = planner call, . = denied before Bedrock)")
print(f"60 moments: {gate.passed} planner calls, {gate.denied} denied before Bedrock")
print("recorded run, same agent:", run["totals"]["A"]["plan_calls"], "plan calls")

P..........PPP.P.......PP........P....P...PPPPP...P.....P.P.   (P = planner call, . = denied before Bedrock)
60 moments: 17 planner calls, 43 denied before Bedrock
recorded run, same agent: 17 plan calls


## 5. Results of the final run, recomputed from the raw rows

3 scripted matches, 180 moments, 173 with an answer key. Cost is Amazon Bedrock only (Decider ran locally).

In [6]:
bench = json.loads((ROOT / "docs/results/bench_cli_20261024_023923.json").read_text())
rows = [r for r in bench["rows"] if r["suite"] == "matches"]
names = {"A": "Decider decides", "A2": "Decider + Nova Pro when unsure", "B1": "Nova Micro every moment", "B2": "Nova Pro decides + plans every moment"}
lines = ["| Agent | Right tactic | Median decision | Within 1 s | Bedrock calls / match | Bedrock cost / match |", "|---|---|---|---|---|---|"]
for f, name in names.items():
    rs = [r for r in rows if r["flow"] == f]
    lab = [r for r in rs if r["reference_policy"]]
    acc = sum(r["effective_policy"] == r["reference_policy"] for r in lab) / len(lab)
    lat = statistics.median(r["decision_latency_ms"] for r in rs) / 1000
    fast = sum(r["decision_latency_ms"] <= 1000 for r in rs) / len(rs)
    n_matches = len({r["match_id"] for r in rs})
    calls = sum(r["bedrock_calls"] for r in rs) / n_matches
    cost = sum(r["cost_usd"] for r in rs) / n_matches
    lines.append(f"| {name} | {acc:.1%} | {lat:.2f} s | {fast:.0%} | {calls:.1f} | ${cost:.3f} |")
display(Markdown("\n".join(lines)))

| Agent | Right tactic | Median decision | Within 1 s | Bedrock calls / match | Bedrock cost / match |
|---|---|---|---|---|---|
| Decider decides | 69.4% | 0.37 s | 100% | 15.7 | $0.036 |
| Decider + Nova Pro when unsure | 82.1% | 0.76 s | 53% | 44.7 | $0.068 |
| Nova Micro every moment | 48.6% | 1.01 s | 42% | 79.3 | $0.048 |
| Nova Pro decides + plans every moment | 79.2% | 3.98 s | 0% | 60.7 | $0.154 |

## 6. Decider's confidence is a usable signal

When Decider is unsure, its pick is right much less often, which is what lets the *"Nova Pro when unsure"* variant hand exactly those moments over.

In [7]:
a2 = [r for r in rows if r["flow"] == "A2" and r["reference_policy"]]
for label, sel in [("confident (>= 0.30)", [r for r in a2 if not r["fallback"]]), ("unsure (< 0.30)", [r for r in a2 if r["fallback"]])]:
    right = sum(r["raw_decider_policy"] == r["reference_policy"] for r in sel)
    print(f"{label:20} Decider right {right}/{len(sel)} = {right / len(sel):.0%}")

confident (>= 0.30)  Decider right 82/96 = 85%
unsure (< 0.30)      Decider right 38/77 = 49%


## 7. Optional: live calls

Set `RUN_LIVE = True` with `APP_MODE=real`, a Strands Decider endpoint (`DECIDER_MODE=hosted` plus `DECIDER_API_KEY`, or `DECIDER_MODE=local`) and AWS credentials with Bedrock access to Amazon Nova. This makes one Decider call on the state above (no Bedrock cost).

In [8]:
RUN_LIVE = False
if RUN_LIVE:
    from matchsense.decider_client import DeciderClient
    client = DeciderClient()
    print(client.health())
    r = client.decide_choice(state_to_text(state), QUESTION, POLICY_OPTIONS)
    print(r.choice, r.confidence, r.client_latency_ms, "ms")
else:
    print("Live calls skipped (RUN_LIVE = False). Everything above used the committed real runs.")

Live calls skipped (RUN_LIVE = False). Everything above used the committed real runs.
